# 06 — Fan-Out and Consensus: Parallel Agent Reasoning

## What is Fan-Out?

**Fan-out** launches multiple agent nodes in **parallel** against the same context, then merges their results using a **consensus strategy**. This is ideal for:

- **Scenario analysis** — bear/base/bull forecasts
- **Ensemble decisions** — multiple models vote on a classification
- **Peer review** — multiple critics evaluate the same artifact

```
                    ┌─── EchoAgent (bear_scenario) ───┐
  running_workflow ─┼─── EchoAgent (base_scenario) ───┼─► consensus → merged_result
                    └─── EchoAgent (bull_scenario) ───┘
```

## Consensus Strategies

| Strategy | Description |
|----------|-------------|
| `highest_confidence` | Pick the node output with the highest `confidence` field |
| `aggregate` | Merge all outputs into a combined result |
| `majority_vote` | Use the most common answer across nodes |
| `first_success` | Use the first node to complete successfully |

## Notebook Goal

1. Start a simple 2-node workflow (parse → echo)
2. After launch, send a `fan_out` signal with 3 parallel `EchoAgent` nodes representing bear/base/bull scenarios
3. Demonstrate all 4 consensus strategies
4. Inspect `fan_outs_executed` in metrics
5. Visualize which consensus strategy won

In [ ]:
import time
import json
import pprint

from multigen import SyncMultigenClient, FanOutRequest, FanOutNodeDef
from multigen.dsl import GraphBuilder, WorkflowBuilder

ORCHESTRATOR_URL = "http://localhost:8009"
client = SyncMultigenClient(base_url=ORCHESTRATOR_URL, timeout=60.0)
assert client.ping(), "Orchestrator not reachable."
print("Connected.")

## 1. Build a Simple 2-Node Base Workflow

The base workflow parses a resume, then produces an initial assessment. The fan-out will be injected after launch to perform parallel scenario analysis.

In [ ]:
graph_def = (
    GraphBuilder()
    .node("parse_resume")
        .agent("ResumeParserAgent")
        .params(output_format="json")
        .timeout(30)
        .done()
    .node("initial_assessment")
        .agent("EchoAgent")
        .params(message="Initial assessment placeholder")
        .timeout(15)
        .done()
    .edge("parse_resume", "initial_assessment")
    .entry("parse_resume")
    .max_cycles(10)
    .build()
)

payload = {
    "resume_text": "David Kim, 6 years quant finance, Python, numpy, pandas, risk modeling.",
    "candidate_id": "candidate-005",
}

response = client.run_graph(graph_def=graph_def, payload=payload)
instance_id = response.instance_id
print(f"Base workflow launched — instance_id: {instance_id}")

## 2. Understanding FanOutRequest

```python
FanOutRequest(
    group_id="my_fan_out",          # unique identifier for this fan-out group
    consensus="highest_confidence", # how to merge results
    nodes=[
        FanOutNodeDef(id="node_a", agent="AgentA", params={...}),
        FanOutNodeDef(id="node_b", agent="AgentB", params={...}),
    ]
)
```

The `group_id` is used to track the fan-out group in metrics and state. All nodes in a group run concurrently.

## 3. Send Fan-Out Signal — Bear/Base/Bull Scenarios

Each scenario is a separate `EchoAgent` invocation with different parameter sets representing pessimistic (bear), neutral (base), and optimistic (bull) assessments of the candidate.

In [ ]:
# Allow the base workflow to start before sending the fan-out
time.sleep(2)

fan_out_req = FanOutRequest(
    group_id="scenario_analysis",
    consensus="highest_confidence",   # pick the most confident scenario result
    nodes=[
        FanOutNodeDef(
            id="bear_scenario",
            agent="EchoAgent",
            params={
                "scenario": "bear",
                "message": "Pessimistic assessment: candidate skill gaps are significant.",
                "confidence": 0.55,
            },
        ),
        FanOutNodeDef(
            id="base_scenario",
            agent="EchoAgent",
            params={
                "scenario": "base",
                "message": "Neutral assessment: candidate meets most requirements.",
                "confidence": 0.78,
            },
        ),
        FanOutNodeDef(
            id="bull_scenario",
            agent="EchoAgent",
            params={
                "scenario": "bull",
                "message": "Optimistic assessment: candidate exceeds requirements.",
                "confidence": 0.91,
            },
        ),
    ],
)

fan_out_result = client.fan_out(instance_id, fan_out_req)
print("Fan-out sent:")
pprint.pprint(fan_out_result)

## 4. Consensus Strategies — Conceptual Overview

Here we explain what each strategy would do with our 3 scenario outputs:

| Strategy | Our 3 Nodes | Result |
|----------|------------|--------|
| `highest_confidence` | bear=0.55, base=0.78, bull=0.91 | **bull_scenario** (0.91 wins) |
| `aggregate` | all 3 outputs | merged dict of all 3 |
| `majority_vote` | all say different things | depends on matching values |
| `first_success` | whichever finishes first | race condition winner |

In [ ]:
# Simulate the consensus decision locally for illustration
scenarios = [
    {"id": "bear_scenario", "scenario": "bear", "confidence": 0.55},
    {"id": "base_scenario", "scenario": "base", "confidence": 0.78},
    {"id": "bull_scenario", "scenario": "bull", "confidence": 0.91},
]

def show_consensus(strategy, scenarios):
    print(f"\nStrategy: {strategy}")
    print("-" * 40)
    if strategy == "highest_confidence":
        winner = max(scenarios, key=lambda s: s["confidence"])
        print(f"  Winner: {winner['id']} (confidence={winner['confidence']})")
    elif strategy == "aggregate":
        aggregated = {s["id"]: s["confidence"] for s in scenarios}
        avg = sum(s["confidence"] for s in scenarios) / len(scenarios)
        print(f"  Merged scores: {aggregated}")
        print(f"  Average confidence: {avg:.3f}")
    elif strategy == "majority_vote":
        # In a real system, votes are on discrete outcomes
        print("  Each node votes: bear=reject, base=maybe, bull=accept")
        print("  No majority → fall back to highest_confidence")
    elif strategy == "first_success":
        print("  First node to complete wins (non-deterministic in practice)")
        print(f"  Assume: {scenarios[1]['id']} finished first")

for strategy in ["highest_confidence", "aggregate", "majority_vote", "first_success"]:
    show_consensus(strategy, scenarios)

## 5. Wait and Check Fan-Out Results

In [ ]:
# Give nodes time to execute
time.sleep(5)

state = client.get_state(instance_id)
metrics = client.get_metrics(instance_id)

print(f"Workflow state — {state.count} node output(s):\n")
for ns in state.nodes:
    print(f"  Node: {ns.node_id}")
    if ns.output:
        conf = ns.output.get("confidence", "n/a")
        print(f"    confidence: {conf}")

print(f"\nMetrics:")
print(f"  nodes_executed    : {metrics.nodes_executed}")
print(f"  fan_outs_executed : {metrics.fan_outs_executed}")

## 6. Visualize Scenario Confidence Scores

A bar chart comparing the confidence of each parallel scenario.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    scenario_names = [s["id"] for s in scenarios]
    confidences = [s["confidence"] for s in scenarios]
    colors = ["#d32f2f", "#fbc02d", "#388e3c"]  # red=bear, yellow=base, green=bull

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(scenario_names, confidences, color=colors, edgecolor="black", linewidth=0.8)

    # Annotate bars
    for bar, conf in zip(bars, confidences):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{conf:.2f}",
            ha="center", va="bottom", fontweight="bold", fontsize=12,
        )

    # Highlight the winner
    winner_idx = confidences.index(max(confidences))
    bars[winner_idx].set_edgecolor("gold")
    bars[winner_idx].set_linewidth(3)

    ax.set_ylim(0, 1.15)
    ax.axhline(y=max(confidences), color="gold", linestyle="--", linewidth=1.5, label=f"Winner threshold ({max(confidences)})")
    ax.set_xlabel("Fan-Out Scenarios", fontsize=12)
    ax.set_ylabel("Confidence Score", fontsize=12)
    ax.set_title("Fan-Out Consensus: Scenario Confidence Comparison\n(highest_confidence strategy)", fontsize=13)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig("fan_out_consensus.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Chart saved to fan_out_consensus.png")
except ImportError:
    print("matplotlib not installed — skipping chart. Install with: pip install matplotlib")

## 7. Multiple Fan-Outs — Using Different Consensus Strategies

In [ ]:
# Launch a fresh workflow to demonstrate aggregate consensus
response2 = client.run_graph(graph_def=graph_def, payload=payload)
instance_id2 = response2.instance_id
time.sleep(2)

aggregate_req = FanOutRequest(
    group_id="aggregate_analysis",
    consensus="aggregate",   # merge all outputs
    nodes=[
        FanOutNodeDef(id="view_a", agent="EchoAgent", params={"perspective": "technical", "score": 0.8}),
        FanOutNodeDef(id="view_b", agent="EchoAgent", params={"perspective": "cultural_fit", "score": 0.7}),
        FanOutNodeDef(id="view_c", agent="EchoAgent", params={"perspective": "leadership", "score": 0.65}),
    ],
)

result2 = client.fan_out(instance_id2, aggregate_req)
print(f"Aggregate fan-out sent for instance {instance_id2}: {result2}")

time.sleep(4)
m2 = client.get_metrics(instance_id2)
print(f"fan_outs_executed (instance 2): {m2.fan_outs_executed}")

In [ ]:
client.close()
print("Done.")

---

## Summary

```python
# Fan-out signal — parallel execution with consensus
client.fan_out(
    instance_id,
    FanOutRequest(
        group_id="my_group",
        consensus="highest_confidence",  # or aggregate / majority_vote / first_success
        nodes=[
            FanOutNodeDef(id="node_a", agent="AgentA", params={...}),
            FanOutNodeDef(id="node_b", agent="AgentB", params={...}),
        ]
    )
)
```

**Next**: See `07_dynamic_agents.ipynb` to learn about blueprint-based dynamic agent creation.